In [1]:
# =========================
# 0) (필요 시) 패키지 설치 확인
# =========================
# 주의: 시험 중 설치는 시간이 걸릴 수 있음. 가능한 최소로!
# !mamba list transformers
!mamba install -y transformers datasets evaluate
!mamba install -c conda-forge scikit-learn
!mamba install -y -c conda-forge "accelerate"

import os
import random
import numpy as np

import torch

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, 
                        TrainingArguments, Trainer)

import evaluate  # metric 계산용(accuracy 등)

C:\Users\uqpua\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------------
# 1) 랜덤 시드 고정(재현성)
# -------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# -------------------------
# 2) IMDB 데이터 로드
# -------------------------
def load_imdb_subset(train_size=200, val_size=100):
    """
    IMDB는 Hugging Face에 내장되어 있어 바로 로드 가능.
    CPU-only에서는 전체 학습이 너무 느리니 일부만 사용.
    """
    ds = load_dataset("imdb")  # ds["train"], ds["test"] 존재

    # train 일부 추출
    train_ds = ds["train"].shuffle(seed=42).select(range(train_size))

    # IMDB는 기본적으로 validation split이 없어서 test를 validation으로 씀(시험/샘플용)
    val_ds = ds["test"].shuffle(seed=42).select(range(val_size))

    return train_ds, val_ds

In [3]:
# -------------------------
# 3) 토크나이징(텍스트 -> 토큰)
# -------------------------
def tokenize_dataset(train_ds, val_ds, tokenizer, max_length=256):
    """
    IMDB 컬럼:
    - text: 리뷰 문자열
    - label: 0(negative) / 1(positive)
    """
    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            truncation=True,          # 길면 자름
            padding="max_length",     # 길이 고정(초심자 안정)
            max_length=max_length
        )

    # batched=True -> 여러 샘플을 한 번에 처리(빠름)
    train_tok = train_ds.map(tokenize_fn, batched=True)
    val_tok = val_ds.map(tokenize_fn, batched=True)

    # Trainer가 torch 텐서를 쓰도록 지정
    train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    return train_tok, val_tok

In [4]:
# =========================
# 3) metric(정확도) 계산 함수
# =========================
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """
    Trainer가 평가할 때 호출하는 함수.
    eval_pred = (logits, labels)
    logits: 모델 출력(클래스별 점수)
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)  # 가장 큰 값의 클래스가 예측
    return accuracy.compute(predictions=preds, references=labels)

In [5]:
# =========================
# 5) 학습 메인 함수
# =========================
def train_basic_text_classification(
    model_name="distilbert-base-uncased",
    output_dir="./basic_ckpt",
    num_labels = 2
):
    """
    1) tokenizer/model 로드
    2) dataset 토크나이즈
    3) Trainer로 학습
    4) 검증 정확도 출력
    """

    # (1) seed 고정
    set_seed(42)

    # (2) CPU 환경임을 명시(시험 환경이 GPU 없음)
    device = torch.device("cpu")
    print("Device:", device)
    print("CUDA available?", torch.cuda.is_available())

    # (3) tokenizer 로드
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # (4) 분류 모델 로드 (num_labels는 클래스 수)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    model.to(device)

    # (5) 데이터 로드(일부만)+ 토큰화
    train_ds, val_ds = load_imdb_subset(train_size=2000, val_size=1000)
    train_tok, val_tok = tokenize_dataset(train_ds, val_ds, tokenizer, max_length=256)
    # train_ds, val_ds = load_and_tokenize_dataset(train_csv, val_csv, tokenizer, max_length=128)

    # (6) Trainer 설정
    # CPU-only에서는 batch_size를 너무 크게 하면 느려지고 메모리도 부담됨
    args = TrainingArguments(
        output_dir=output_dir,              # 체크포인트 저장 폴더
        eval_strategy="epoch",              # epoch 끝마다 평가, eval_strategy, evaluation_strategy
        save_strategy="epoch",                 # epoch 끝마다 저장, no 불필요
        logging_strategy="steps",           # steps 단위로 로그
        logging_steps=50,
        learning_rate=2e-5,                 # 기본적으로 안정적인 LR
        per_device_train_batch_size=8,      # CPU에서 무난(느리면 4로)
        per_device_eval_batch_size=16,
        num_train_epochs=2,                 # 시험에서는 1~3 정도가 현실적
        weight_decay=0.01,
        load_best_model_at_end=True,        # 가장 좋은 평가 점수 모델 로드
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none"                    # wandb 등 외부 로깅 끔(시험 안전)
    )

    # (7) Trainer 생성
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    # (8) 학습 시작
    trainer.train()

    # (9) 최종 평가
    metrics = trainer.evaluate()
    print("Final Eval Metrics:", metrics)

    # (10) 필요하다면 모델 저장(보통 시험 제출엔 불필요)
    # 단, 제출 파일 20MB 제한이 있으므로 저장 결과를 제출에 포함하면 안 됨!
    # trainer.save_model(output_dir)

    return metrics


In [6]:
# =========================
# 5) 실행
# =========================
# num_labels는 데이터 라벨 개수에 맞게 바꿔야 함(예: 3-class면 3)
# train_basic_text_classification(num_labels=3)

# 기본은 2-class라고 가정하고 실행
# (실제 시험에서는 파일명/컬럼명만 맞추면 됨)
train_basic_text_classification()

Device: cpu
CUDA available? False


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the latest cached version of the dataset since imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\uqpua\.cache\huggingface\datasets\imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Sun Jan 25 11:17:52 2026).
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_3844\4111273625.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 